In [1]:
import hashlib
import os
import random 
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import pandas as pd

In [2]:
class ECGAdvancedConcatenator:
    """
    Advanced ECG data concatenator with sequential HR ordering and duration-aware truncation.
    """

    def __init__(self, csv_label_file: Optional[str], data_dir: str, labels: Optional[List[int]] = None) -> None:
        if csv_label_file is not None and not os.path.isfile(csv_label_file):
            raise FileNotFoundError(f"CSV label file not found: {csv_label_file}")
        if not os.path.isdir(data_dir):
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

        self.csv_label_file = csv_label_file
        self.data_dir = data_dir
        self.labels = labels or [0, 1, 2, 3]
        self.label_files: Dict[int, List[str]] = {}
        self.data_cache: Dict[str, pd.DataFrame] = {}

        if self.csv_label_file:
            self._load_label_mapping()
        else:
            self._scan_label_directories()

    def _scan_label_directories(self) -> None:
        for label in self.labels:
            label_dir = os.path.join(self.data_dir, str(label))
            if not os.path.isdir(label_dir):
                raise FileNotFoundError(f"Label directory not found: {label_dir}")
            files = [f for f in os.listdir(label_dir) if f.lower().endswith(".csv")]
            if not files:
                raise FileNotFoundError(f"No CSV files found in {label_dir}")
            self.label_files[label] = sorted(files, key=self._extract_hr_number)

    def _load_label_mapping(self) -> None:
        df = pd.read_csv(self.csv_label_file)
        df.columns = [c.strip() for c in df.columns]
        if "File" not in df.columns or "Label" not in df.columns:
            raise ValueError("CSV must include 'File' and 'Label' columns.")

        for _, row in df.iterrows():
            label = int(row["Label"])
            filename = str(row["File"])
            self.label_files.setdefault(label, []).append(filename)

        for label in self.label_files:
            self.label_files[label] = sorted(self.label_files[label], key=self._extract_hr_number)

    def _get_full_path(self, label: int, filename: str) -> str:
        return os.path.join(self.data_dir, str(label), filename)

    def _load_label_files(self, label: int) -> None:
        if label not in self.label_files:
            raise ValueError(f"Label {label} is not available.")

        for filename in self.label_files[label]:
            full_path = self._get_full_path(label, filename)
            if full_path in self.data_cache:
                continue
            if not os.path.isfile(full_path):
                raise FileNotFoundError(f"File missing for label {label}: {full_path}")
            df = pd.read_csv(full_path)
            df.columns = [c.strip() for c in df.columns]
            self.data_cache[full_path] = df

    @staticmethod
    def _get_duration_from_dataframe(df: pd.DataFrame) -> float:
        if "Time" in df.columns:
            time_series = df["Time"].to_numpy()
            if len(time_series) == 0:
                return 0.0
            return float(time_series[-1] - time_series[0])
        return float(len(df))

    @staticmethod
    def _extract_hr_number(filename: str) -> int:
        base = os.path.splitext(os.path.basename(filename))[0].lower()
        # Support names like hr80.csv and hr80_1.csv by reading the first hr<number> token.
        m = re.search(r"hr(\d+)", base)
        if m:
            return int(m.group(1))

        # Fallback: only parse the first numeric token before '_' to avoid hr80_1 -> 801.
        first_token = base.split("_", 1)[0]
        m2 = re.search(r"(\d+)", first_token)
        if m2:
            return int(m2.group(1))

        raise ValueError(f"Unable to extract HR number from filename: {filename}")

    @staticmethod
    def _offset_time(df: pd.DataFrame, offset: float) -> pd.DataFrame:
        if "Time" not in df.columns:
            return df
        df = df.copy()
        df["Time"] = df["Time"] + offset
        return df

    @staticmethod
    def _get_next_file_index(directory: str, prefix: str) -> int:
        max_index = 0
        if os.path.isdir(directory):
            for name in os.listdir(directory):
                if not name.lower().endswith(".csv"):
                    continue
                stem = os.path.splitext(name)[0]
                token = f"{prefix}_"
                if not stem.startswith(token):
                    continue
                suffix = stem[len(token):]
                if suffix.isdigit():
                    max_index = max(max_index, int(suffix))
        return max_index + 1

    @staticmethod
    def _hash_dataframe(df: pd.DataFrame) -> str:
        csv_bytes = df.to_csv(index=False).encode("utf-8")
        return hashlib.sha256(csv_bytes).hexdigest()

    @staticmethod
    def _collect_hashes_in_folder(folder: str) -> set:
        hashes = set()
        if not os.path.isdir(folder):
            return hashes
        for name in os.listdir(folder):
            if not name.lower().endswith(".csv"):
                continue
            fp = os.path.join(folder, name)
            with open(fp, "rb") as fh:
                hashes.add(hashlib.sha256(fh.read()).hexdigest())
        return hashes

    def concatenate_preserve_time(self, label: int, duration_minutes: float, random_order: bool = True) -> pd.DataFrame:
        self._load_label_files(label)
        label_files = list(self.label_files[label])
        if random_order:
            random.shuffle(label_files)
        else:
            label_files = sorted(label_files, key=self._extract_hr_number)

        target_seconds = duration_minutes * 60.0
        total_duration = 0.0
        output_parts = []
        time_offset = 0.0

        for filename in label_files:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)

            remaining = target_seconds - total_duration
            if remaining <= 0:
                break

            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            # Truncate last file to match the remaining duration.
            truncated = df.copy()
            if "Time" in truncated.columns:
                start_time = truncated["Time"].iloc[0]
                cutoff = start_time + remaining
                truncated = truncated[truncated["Time"] <= cutoff]
                if len(truncated) > 0:
                    truncated["Time"] = truncated["Time"] - start_time + time_offset
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(truncated)
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError(f"No data available for label {label}.")

        return pd.concat(output_parts, ignore_index=True)

    def concatenate_sequential_hr(self, labels: List[int], num_segments: int, output_dir: str, target_minutes: float) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if num_segments <= 0:
            raise ValueError("num_segments must be positive.")

        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        labeled_files.sort(key=lambda item: (self._extract_hr_number(item[1]), item[0]))

        target_seconds = target_minutes * 60.0
        segments: List[Tuple[int, str]] = []

        for label, filename in labeled_files:
            segments.append((label, filename))
            if len(segments) < num_segments:
                continue

            output_df, majority_label = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_prefix = f"mixed_{majority_label}"
            else:
                final_dir = output_dir
                file_prefix = f"concat_{majority_label}"
            os.makedirs(final_dir, exist_ok=True)
            next_index = self._get_next_file_index(final_dir, file_prefix)
            file_name = f"{file_prefix}_{next_index:03d}.csv"

            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            segments = []

    def concatenate_random_hr(
        self,
        labels: List[int],
        files_per_segment: int,
        n_outputs: int,
        output_dir: str,
        target_minutes: float,
        allow_replacement: bool = True,
        random_seed: Optional[int] = None,
        files_per_segment_max: Optional[int] = None,
        required_majority_label: Optional[int] = None,
        min_majority_ratio: float = 0.50,
        hr_band_by_label: Optional[Dict[int, Tuple[int, int]]] = None,
        choose_unique_hr_variants: bool = True,
        adaptive_relaxation: bool = True,
    ) -> None:
        if not labels:
            raise ValueError("Labels list cannot be empty.")
        if files_per_segment <= 0:
            raise ValueError("files_per_segment must be positive.")
        if files_per_segment_max is None:
            files_per_segment_max = files_per_segment
        if files_per_segment_max < files_per_segment:
            raise ValueError("files_per_segment_max must be >= files_per_segment.")
        if n_outputs <= 0:
            raise ValueError("n_outputs must be positive.")

        rng = random.Random(random_seed)
        os.makedirs(output_dir, exist_ok=True)

        labeled_files: List[Tuple[int, str]] = []
        for label in labels:
            self._load_label_files(label)
            for filename in self.label_files[label]:
                labeled_files.append((label, filename))

        if not labeled_files:
            raise ValueError("No files available for selected labels.")

        # Group hrX/hrX_1... variants and sample one file per HR key for better diversity.
        hr_group_pool: List[Tuple[int, int, List[str]]] = []
        if choose_unique_hr_variants:
            grouped: Dict[Tuple[int, int], List[str]] = {}
            for label, filename in labeled_files:
                hr_num = self._extract_hr_number(filename)
                grouped.setdefault((label, hr_num), []).append(filename)
            hr_group_pool = [
                (label, hr_num, sorted(variants))
                for (label, hr_num), variants in grouped.items()
            ]
            if not hr_group_pool:
                raise ValueError("No HR groups available for random concatenation.")
            if files_per_segment > len(hr_group_pool):
                raise ValueError("files_per_segment is larger than number of unique HR groups.")
        if not allow_replacement and len(labeled_files) < files_per_segment:
            raise ValueError("Not enough files for sampling without replacement.")

        target_seconds = target_minutes * 60.0
        created = 0
        attempts = 0
        max_attempts = max(n_outputs * 50, 200)
        stall_attempts = 0
        max_stall_attempts = max(400, n_outputs * 10)
        current_min_majority_ratio = float(min_majority_ratio)
        current_hr_band_padding = 0.0
        reject_stats = Counter()
        existing_hashes_cache: Dict[str, set] = {}

        while created < n_outputs and attempts < max_attempts:
            attempts += 1
            if adaptive_relaxation and stall_attempts >= max_stall_attempts:
                old_ratio = current_min_majority_ratio
                current_min_majority_ratio = max(0.55, current_min_majority_ratio - 0.02)
                current_hr_band_padding = min(3.0, current_hr_band_padding + 0.5)
                stall_attempts = 0
                print(
                    f"Adaptive relaxation -> min_majority_ratio: {old_ratio:.2f} -> {current_min_majority_ratio:.2f}, "
                    f"hr_band_padding: +/-{current_hr_band_padding:.1f}"
                )
            n_files_this_output = rng.randint(files_per_segment, files_per_segment_max)
            if choose_unique_hr_variants:
                n_pick = min(n_files_this_output, len(hr_group_pool))
                selected_groups = rng.sample(hr_group_pool, n_pick)
                segments = [
                    (label, rng.choice(variants))
                    for label, _, variants in selected_groups
                ]
            else:
                if not allow_replacement and len(labeled_files) < n_files_this_output:
                    raise ValueError("Not enough files for this output when sampling without replacement.")
                if allow_replacement:
                    segments = [rng.choice(labeled_files) for _ in range(n_files_this_output)]
                else:
                    segments = rng.sample(labeled_files, n_files_this_output)

            label_counts = Counter([label for label, _ in segments])
            majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
            majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))
            if required_majority_label is not None and majority_label != required_majority_label:
                reject_stats['majority_label_mismatch'] += 1
                stall_attempts += 1
                continue
            if majority_ratio < current_min_majority_ratio:
                reject_stats['majority_ratio'] += 1
                stall_attempts += 1
                continue
            if hr_band_by_label:
                band = hr_band_by_label.get(majority_label)
                if band is not None:
                    hr_values_majority = [
                        self._extract_hr_number(filename)
                        for label, filename in segments
                        if label == majority_label
                    ]
                    if not hr_values_majority:
                        reject_stats['empty_majority_hr_values'] += 1
                        stall_attempts += 1
                        continue
                    mean_hr_majority = sum(hr_values_majority) / len(hr_values_majority)
                    low = band[0] - current_hr_band_padding
                    high = band[1] + current_hr_band_padding
                    if mean_hr_majority < low or mean_hr_majority > high:
                        reject_stats['hr_band'] += 1
                        stall_attempts += 1
                        continue

            output_df, _, _ = self._build_duration_segment(segments, target_seconds)
            if len(labels) > 1:
                final_dir = os.path.join(output_dir, f"mixed_{majority_label}")
                file_prefix = f"mixed_{majority_label}"
            else:
                final_dir = output_dir
                file_prefix = f"concat_{majority_label}"
            os.makedirs(final_dir, exist_ok=True)

            if final_dir not in existing_hashes_cache:
                existing_hashes_cache[final_dir] = self._collect_hashes_in_folder(final_dir)

            content_hash = self._hash_dataframe(output_df)
            if content_hash in existing_hashes_cache[final_dir]:
                reject_stats['duplicate_hash'] += 1
                stall_attempts += 1
                continue

            next_index = self._get_next_file_index(final_dir, file_prefix)
            file_name = f"{file_prefix}_{next_index:03d}.csv"
            output_path = os.path.join(final_dir, file_name)
            output_df.to_csv(output_path, index=False)

            existing_hashes_cache[final_dir].add(content_hash)
            created += 1
            stall_attempts = 0

        print(f"Random generation done: created={created}, attempts={attempts}, skipped={attempts - created}")
        if reject_stats:
            print(f"Reject stats: {dict(reject_stats)}")
        if created < n_outputs:
            print("Warning: unable to create all requested unique outputs with current constraints.")

    def _build_duration_segment(
        self,
        segments: List[Tuple[int, str]],
        target_seconds: float,
        *,
        rng: Optional[random.Random] = None,
        randomize_truncation: bool = False,
    ) -> Tuple[pd.DataFrame, int, float]:
        label_counts = Counter([label for label, _ in segments])
        majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
        majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))

        output_parts = []
        total_duration = 0.0
        time_offset = 0.0

        for label, filename in segments:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)
            remaining = target_seconds - total_duration
            if remaining <= 0:
                break
            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            truncated = df.copy()
            if "Time" in truncated.columns and len(truncated) > 0:
                start_time = float(truncated["Time"].iloc[0])
                end_time = float(truncated["Time"].iloc[-1])
                duration = end_time - start_time
                if randomize_truncation and rng is not None and duration > remaining:
                    max_shift = max(0.0, duration - remaining)
                    window_start = start_time + (rng.random() * max_shift)
                    window_end = window_start + remaining
                    truncated = truncated[(truncated["Time"] >= window_start) & (truncated["Time"] <= window_end)]
                    if len(truncated) == 0:
                        truncated = df[df["Time"] <= window_end].tail(1)
                    truncated = truncated.copy()
                    truncated["Time"] = truncated["Time"] - float(truncated["Time"].iloc[0])
                else:
                    cutoff = start_time + remaining
                    truncated = truncated[truncated["Time"] <= cutoff]
                    if len(truncated) > 0:
                        truncated = truncated.copy()
                        truncated["Time"] = truncated["Time"] - float(truncated["Time"].iloc[0])
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(self._offset_time(truncated, time_offset))
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError("Unable to build concatenated segment from provided files.")

        return pd.concat(output_parts, ignore_index=True), majority_label, majority_ratio

In [ ]:
from pathlib import Path
import os
import random
import sys
from collections import Counter

import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from features import extract_features_from_window

# ---- Config ----
RAW_BASE_DIR = str(DATA_DIR / "raw_gen")
OUTPUT_ROOT_DIR = str(DATA_DIR / "concatenated")
DATASET_ID = 5
FEATURES_DIR = str(DATA_DIR / "features" / str(DATASET_ID))

LABELS_FOR_ORDER = [0, 1, 2, 3]
ORDER_REPEATS_PER_LABEL = 4
SHUFFLE_SEED = None

LABEL_ORDER = []
for lb in LABELS_FOR_ORDER:
    LABEL_ORDER.extend([lb] * int(ORDER_REPEATS_PER_LABEL))
random.Random(SHUFFLE_SEED).shuffle(LABEL_ORDER)

TARGET_HOURS_PER_LEVEL = 4.0
SEGMENT_SECONDS = 300.0  # smaller segment => more interleaving

WINDOW_SECONDS = 256.0
STEP_SECONDS = 256.0
RANDOM_SEED = None

# ---- Helpers ----

def _build_balanced_plan(order, target_seconds_per_label, segment_seconds):
    remaining = {label: float(target_seconds_per_label) for label in set(order)}
    plan = []
    idx = 0
    while any(v > 0 for v in remaining.values()):
        label = order[idx % len(order)]
        idx += 1
        if remaining[label] <= 0:
            continue
        seg = min(float(segment_seconds), remaining[label])
        plan.append((label, seg))
        remaining[label] -= seg
    return plan


def _pick_segments_for_label(concat, label, target_seconds, rng):
    concat._load_label_files(label)
    files = list(concat.label_files.get(label, []))
    if not files:
        raise ValueError(f"No files available for label={label}")

    segments = []
    total = 0.0
    safety = 0
    while total < target_seconds:
        safety += 1
        if safety > 20000:
            raise RuntimeError("Safety stop: too many iterations while building segment.")

        filename = rng.choice(files)
        full_path = concat._get_full_path(label, filename)
        df = concat.data_cache[full_path]
        duration = concat._get_duration_from_dataframe(df)
        if duration <= 0:
            continue

        segments.append((label, filename))
        total += duration

    return segments


def _assign_labels_by_bounds(df, bounds):
    labels = []
    for t in df["Time"]:
        found = False
        for start, end, label in bounds:
            if start <= t < end or (abs(t - end) < 1e-6 and t == df["Time"].iloc[-1]):
                labels.append(label)
                found = True
                break
        if not found:
            labels.append(None)
    out = df.copy()
    out["label"] = labels
    return out


def _extract_features_windowed(ecg_df, window_seconds, step_seconds, source_name):
    if "Time" not in ecg_df.columns:
        raise ValueError("ecg_df must contain a 'Time' column.")

    t_min = float(ecg_df["Time"].min())
    t_max = float(ecg_df["Time"].max())
    rows = []
    window_id = 0
    t_start = t_min

    while t_start + window_seconds <= t_max + 1e-6:
        t_end = t_start + window_seconds
        window_df = ecg_df[(ecg_df["Time"] >= t_start) & (ecg_df["Time"] < t_end)].copy()
        feat = extract_features_from_window(
            g=window_df,
            window_id=window_id,
            source_file=source_name,
            window_start=t_start,
            window_end=t_end,
        )
        if feat is not None:
            rows.append(feat)
        window_id += 1
        t_start += step_seconds

    return pd.DataFrame(rows)




FileNotFoundError: Data directory not found: data/raw_gen